# Domino tilings

Consider an $m \times n$ rectangular chessboard. Suppose we want to tile this board with dominoes, where a domino is a $2 \times 1$ rectangle, and a tiling is a way to place several dominoes on the board so that all of its squares are covered, but no dominoes overlap or lie partially on the board.

Is such a tiling possible? If so, how many are there?

Fact: Tilings exist if and only if $m$ and $n$ are not both odd (i.e., $mn$ is even).

Amazingly, there is a closed-form solution:


$$
T(n,m)=2^{\frac{nm}{2}}
\prod_{i=1}^{n}\prod_{j=1}^{m}
\left(
\cos^2\frac{\pi i}{n+1}
+
\cos^2\frac{\pi j}{m+1}
\right)^{\frac{1}{4}}.
$$



In [1]:
import math
import numpy as np
import pandas as pd

def count_tilings(m, n):
    if (m * n) % 2 != 0:
        return 0
    prod = 1.0
    for i in range(1, m+1 ):
        for j in range(1, n+1 ):
            term =  math.sqrt( math.sqrt( (math.cos(i * math.pi / (m + 1))**2) + (math.cos(j * math.pi / (n + 1))**2)) )
            prod *= term
    prod *= 2**(m*n/2)

    return int(round(prod))

In [2]:
M = 20
N = 20
results = {}

In [5]:
for m in range(1, M + 1):
    for n in range(m, N + 1):
        results[(m, n)] = count_tilings(m, n)


df = pd.DataFrame(index=range(1, M + 1), columns=range(1, N + 1))
for (m, n), val in results.items():
    df.loc[m, n] = val
    df.loc[n, m] = val

print(df.to_string())

   1      2       3          4            5               6                 7                    8                      9                         10                          11                             12                               13                                  14                                    15                                       16                                         17                                            18                                              19                                                 20
1   0      1       0          1            0               1                 0                    1                      0                         1                           0                              1                                0                                   1                                     0                                        1                                          0                                             1            


However, floating-point operations make it difficult to calculate $T(n,m)$ accurately using this cosine formula. For example, for the chessboard $M(16 \times 8)$, the result of calculations using the cosine formula, performed by a C# (also python) program using double-precision floating-point real variables, is $540061286536919$, while the true value of $M(16 \times 8)$ is $540061286536921$.

**Rreference**: Magomedov, Abdulkarim M., and Serge A. Lawrence. "An algorithm for counting domino tilings of a rectangular chessboard." Journal of Algebra Combinatorics Discrete Structures and Applications (2026): 15-27.



**Obs. 1:** A closed integer form exists only for $2 \times N$ or $3 \times N$ boards.

For $2 \times N$ boards, the number of tilings follows the Fibonacci sequence. Specifically,

$$
T(2,n)=F_{n+1},
$$

where

$$
F_1=1,\quad F_2=1,\quad F_3=2.
$$

For $3 \times N$ boards, $T(3,2k)=4T(3,2k-2) - T(3,2k-4)$.

**Obs. 2:** For a general $m \times n$ board, we cannot always trust the value computed from the cosine formula, since floating-point operations can affect the final accuracy.

**Obs. 3:** The number of tilings grows exponentially fast? 



### model 1:
Wrong!

In [ ]:
#!/usr/bin/python3
# Category: csplib / tiling

"""
Consider an m x n rectangular chessboard. Suppose we want to tile this board with dominoes,
where a domino is a 2 x 1 rectangle, and a tiling is a way to place several dominoes on the
board so that all of its squares are covered, but no dominoes overlap or lie partially outside
the board.

Print one valid tiling of the board. The solution is represented as an m x n matrix of integers,
where cells with the same integer belong to the same domino.
"""

In [ ]:
# Data
m = 3  # Number of rows
n = 2  # Number of columns
# possible solution for m=3, n=2:
# ||| , =| , |= 
# End of data

# Import libraries
import json
from cpmpy import *

def domino_tiling(m=4, n=5):
    # A domino tiling is only possible if the number of cells is even
    if (m * n) % 2 != 0:
        raise ValueError("No domino tiling exists when m*n is odd.")

    # Each cell gets an integer label.
    # Two cells with the same label form one domino.
    num_dominoes = (m * n) // 2
    board = intvar(1, num_dominoes, shape=(m, n), name="board")

    constraints = []

    # Each label must appear exactly twice
    for d in range(1, num_dominoes + 1):
        constraints.append(sum(board == d) == 2)

    # The two occurrences of each label must be adjacent
    for d in range(1, num_dominoes + 1):
        occ = [(i, j) for i in range(m) for j in range(n)]

        # Boolean indicators for where domino d is placed
        is_d = [[board[i, j] == d for j in range(n)] for i in range(m)]

        # Count adjacent pairs of cells labeled d
        adj_pairs = []
        for i in range(m):
            for j in range(n):
                if i + 1 < m:
                    adj_pairs.append((board[i, j] == d) & (board[i + 1, j] == d))
                if j + 1 < n:
                    adj_pairs.append((board[i, j] == d) & (board[i, j + 1] == d))

        # Exactly one adjacent pair must exist for this domino
        constraints.append(sum(adj_pairs) == 1)

    model = Model(constraints)
    return model, (board,)

# Example usage
model, (board,) = domino_tiling(m, n)

model.solveAll(display=board)


# if model.solve():
#     solution = {"tiling": board.value().tolist()}
# else:
#     solution = {"tiling": None}

# # Print
# print(json.dumps(solution))
# # End of CPMPy script

[[1 3]
 [1 3]
 [2 2]]
[[1 1]
 [3 2]
 [3 2]]
[[1 1]
 [3 3]
 [2 2]]
[[1 1]
 [2 3]
 [2 3]]
[[1 2]
 [1 2]
 [3 3]]
[[1 1]
 [2 2]
 [3 3]]
[[2 1]
 [2 1]
 [3 3]]
[[3 1]
 [3 1]
 [2 2]]
[[3 3]
 [2 1]
 [2 1]]
[[2 2]
 [3 1]
 [3 1]]
[[2 2]
 [1 1]
 [3 3]]
[[3 3]
 [1 1]
 [2 2]]
[[2 2]
 [1 3]
 [1 3]]
[[2 2]
 [3 3]
 [1 1]]
[[3 2]
 [3 2]
 [1 1]]
[[3 3]
 [1 2]
 [1 2]]
[[3 3]
 [2 2]
 [1 1]]
[[2 3]
 [2 3]
 [1 1]]


18

Observations:

1) The flexibility of the problem formulation can lead to an incorrect/undesirable model definition. 
2) Some lines of code are unnecessary. 
3) The code is highly inefficient and becomes expensive even for medium values of 𝑛 and 𝑚. 
4) The size of the solution space is much higher!


### model 2:
Correct!

In [ ]:
#!/usr/bin/python3
# Category: csplib / tiling

"""
Consider an m x n rectangular chessboard. We want to tile this board with dominoes,
where each domino is a 2 x 1 rectangle. A tiling is a placement of dominoes such that
every square of the board is covered exactly once, no dominoes overlap, and no domino
extends beyond the boundary of the board.

Print one valid tiling of the chessboard.
"""

In [1]:
# Data
m = 2 # Number of rows
n = 3  # Number of columns
# End of data

# Import libraries
import json
from cpmpy import *

def domino_tiling(m=4, n=6):
    # A necessary condition for a domino tiling
    if (m * n) % 2 != 0:
        raise ValueError("No domino tiling exists when m*n is odd.")

    # h[i,j] = 1 if a horizontal domino starts at cell (i,j)
    # valid for j = 0..n-2
    h = boolvar(shape=(m, n-1), name="h")

    # v[i,j] = 1 if a vertical domino starts at cell (i,j)
    # valid for i = 0..m-2
    v = boolvar(shape=(m-1, n), name="v")

    constraints = []

    # Every cell must be covered exactly once
    for i in range(m):
        for j in range(n):
            covers = []

            # Horizontal domino starting at (i,j) covers (i,j) and (i,j+1)
            if j < n - 1:
                covers.append(h[i, j])

            # Horizontal domino starting at (i,j-1) covers (i,j)
            if j > 0:
                covers.append(h[i, j-1])

            # Vertical domino starting at (i,j) covers (i,j) and (i+1,j)
            if i < m - 1:
                covers.append(v[i, j])

            # Vertical domino starting at (i-1,j) covers (i,j)
            if i > 0:
                covers.append(v[i-1, j])

            constraints.append(sum(covers) == 1)

    model = Model(constraints)
    return model, (h, v)

# Example usage
model, (h, v) = domino_tiling(m, n)

model.solveAll(display=[h, v] , time_limit=10)
# model.solveAll(time_limit=60*5)

# if model.solve():
#     solution = {
#         "horizontal": h.value().tolist(),
#         "vertical": v.value().tolist()
#     }
# else:
#     solution = {
#         "horizontal": None,
#         "vertical": None
#     }

# # Print
# print(json.dumps(solution))
# # End of CPMPy script

[[[False, True], [False, True]], [[True, False, False]]]
[[[False, False], [False, False]], [[True, True, True]]]
[[[True, False], [True, False]], [[False, False, True]]]


3

## candidates

| ID  | Type    | Category               | Main property                         |
| --- | ------- | ---------------------- | ------------------------------------- |
| C1  | correct | reference              | clean correct general model           |
| C2  | correct | equivalent formulation | correct with parity shortcut          |
| C3  | buggy   | under-constrained      | missing full coverage                 |
| C4  | buggy   | under-constrained      | overlap allowed                       |
| C5  | buggy   | adjacency/boundary     | wrong horizontal domain               |
| C6  | buggy   | over-constrained       | only horizontal dominoes              |
| C7  | buggy   | over-constrained       | only vertical dominoes                |
| C8  | buggy   | feasibility bug        | wrongly requires both dimensions even |
| C9  | buggy   | non-general            | hard-coded to even number of rows     |
| C10 | buggy   | indexing bug           | misses last column coverage logic     |


The right correctness notions

You can define several levels of correctness.

A. Satisfiability equivalence

For each instance:

candidate is SAT iff reference is SAT

This is weakest.

B. Solution validity

For each solution returned by candidate:

it satisfies the true problem semantics

This checks soundness.

C. Solution completeness

For each valid solution of the true problem:

candidate can represent it

This checks completeness.

D. Solution-space equivalence

For each instance:

candidate solution set = reference solution set

For each candidate and instance, annotate:

exact_equivalent
sound_but_incomplete
complete_but_unsound
rare, but possible depending on representation/checking
unsound_and_incomplete
status_only_correct
SAT/UNSAT right, solution space wrong
not_executable

# N-Queens

In [8]:
def solve_n_queens(n):
    def backtrack(row, cols, diag1, diag2):
        if row == n:
            return 1
        
        count = 0
        for col in range(n):
            # Check if column or diagonals are under attack
            # diag1: row - col is constant for \ diagonals
            # diag2: row + col is constant for / diagonals
            if col in cols or (row - col) in diag1 or (row + col) in diag2:
                continue
            
            # Place queen and move to next row
            cols.add(col)
            diag1.add(row - col)
            diag2.add(row + col)
            
            count += backtrack(row + 1, cols, diag1, diag2)
            
            # Backtrack (remove queen)
            cols.remove(col)
            diag1.remove(row - col)
            diag2.remove(row + col)
            
        return count

    return backtrack(0, set(), set(), set())

for n in range(1, 20):  
    print(f"Number of solutions for N={n}: {solve_n_queens(n)}")

Number of solutions for N=1: 1
Number of solutions for N=2: 0
Number of solutions for N=3: 0
Number of solutions for N=4: 2
Number of solutions for N=5: 10
Number of solutions for N=6: 4
Number of solutions for N=7: 40
Number of solutions for N=8: 92
Number of solutions for N=9: 352
Number of solutions for N=10: 724
Number of solutions for N=11: 2680
Number of solutions for N=12: 14200
Number of solutions for N=13: 73712
Number of solutions for N=14: 365596
Number of solutions for N=15: 2279184


KeyboardInterrupt: 

[This part generated by Gemini!]

---

The **N-Queens problem** is a classic combinatorial puzzle where the goal is to place $N$ chess queens on an $N \times N$ chessboard such that no two queens threaten each other. This means no two queens can share the same row, column, or diagonal.

### 1. N-Queens Solution (Backtracking Algorithm)

The most efficient way to solve this for small to medium $N$ is using **backtracking**. 

---

### 2. Solution Space Size (n=4 to n=50)

Finding the number of solutions for the N-Queens problem is a "hard" computational problem (specifically, it is #P-complete). 

**Important Note:** As of 2024, the exact number of solutions is only known up to **$N = 28$**. Calculating $N=50$ is currently impossible with existing global computing power, as the number of solutions grows faster than exponential (super-exponentially).

Below are the known values (Sequence **A000170** in the OEIS):

| N | Total Solutions | N | Total Solutions |
| :--- | :--- | :--- | :--- |
| **4** | 2 | **17** | 95,815,104 |
| **5** | 10 | **18** | 666,090,624 |
| **6** | 4 | **19** | 4,968,057,848 |
| **7** | 40 | **20** | 39,029,188,884 |
| **8** | 92 | **21** | 314,666,222,712 |
| **9** | 352 | **22** | 2,691,008,701,644 |
| **10** | 724 | **23** | 24,233,937,684,440 |
| **11** | 2,680 | **24** | 227,514,171,973,736 |
| **12** | 14,200 | **25** | 2,207,893,435,808,352 |
| **13** | 73,712 | **26** | 22,317,699,616,364,044 |
| **14** | 365,596 | **27** | 234,907,967,154,122,528 |
| **15** | 2,279,184 | **28** | 2,692,442,130,296,474,432 |
| **16** | 14,772,512 | **...** | ... |

### Why can't we calculate $N = 50$?
1.  **Complexity:** The search space for $N=50$ is $50!$ (50 factorial) if we simply try one queen per row, which is roughly $3 \times 10^{64}$. Even with optimized backtracking, the number of valid solutions for $N=50$ is estimated to be around **$10^{40}$**.
2.  **Current Record:** The value for $N=27$ was calculated in 2016 using a massive supercomputing cluster. The value for $N=28$ was only recently verified (2024) by the *N-Queens@Home* project using distributed computing.
3.  **Estimation for $N=50$:** According to Rivin et al., the number of solutions $Q(n)$ behaves approximately as:
    $$Q(n) \approx n! \cdot e^{-cn}$$
    For $N=50$, the result would be a number with dozens of digits, far exceeding the memory and processing capabilities of modern technology to count individually.

If you need a single valid configuration for $N=50$ (rather than the total count), it can be found instantly using a **Greedy Repair algorithm** or **Conflict Minimization** technique.

https://n-queen-five.vercel.app/visualize

In [ ]:
# From DCP-Bench-Open

#!/usr/bin/python3
# Category: csplib
# Source: https://github.com/CPMpy/cpmpy/blob/master/examples/csplib/prob054_n_queens.py
# Source description: https://www.csplib.org/Problems/prob054/

"""
Can \( n \) queens (of the same color) be placed on a \( n \times n \) chessboard so that none of the queens can attack
each other? In chess, a queen attacks other squares on the same row, column, or either diagonal as itself. So the
\( n \)-queens problem is to find a set of \( n \) locations on a chessboard, no two of which are on the same row,
column or diagonal.

Print the positions of the queens on the chessboard (queens) as a list of n integers - ranging from 1 to n, where the i-th
integer represents the column position of the queen in the i-th row.
"""

In [ ]:
# Data
n = 10  # Size of the chessboard and number of queens
# End of data

# Import libraries
import json
import numpy as np
from cpmpy import *

def n_queens(n=8):

    queens = intvar(1, n, shape=n, name="queens")

    # Constraints on columns and left/right diagonal
    model = Model([
        AllDifferent(queens),
        AllDifferent(queens - np.arange(n)),
        AllDifferent(queens + np.arange(n)),
    ])

    return model, (queens,)

# Example usages
model, (queens,) = n_queens(n)
# model.solve()
model.solveAll()

# # Print
# solution = {"queens": queens.value().tolist()}
# print(json.dumps(solution))
# # End of CPMPy script


724

# Discussion:

1) Categories of incorrect samples:

Similar to CodeJudge paper (slide 58): catalog of Severity and Inconsistencies

2) flexible modeling (data n and m can be changed easily) vs. unflexible (like configit/LML models , hard to change the number of variables) 

3) language/library of models? cpmpy and LML ?
cpmpy , PyCSP2, Minizinc , ...

4) how many samples for each toy problem? 10-20? 50-100?

5) Impossible to create complete solution space in large scales